# Begin

In [3]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict # @launchit.collect
import json
import datetime
import pprint
import re
import pickle
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import sqlite3

from tqdm.notebook import tqdm

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
from dataset_utils import *
from samplers import *
from utils import * # @launchit.collect
from logging_utils import *
from image_utils import *
from model_registry import *
from torch_helpers import *
import launchit # @launchit.disable
import optuna_multiprocessing  # @launchit.collect
from hp_utils import *
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement

# Init

In [7]:
ArrayUtils.init()
LOG = Logging.get()
RNG = np.random.default_rng()

# ModelRegistry: 15_* -> 15a_*

In [15]:
model_registry = ModelRegistry('com.develorium.neurovision.15_transformer')
models = model_registry.list_models() # aka components in maven
old_log_level = LOG.set_log_level('all', logging.ERROR)

try:
    for model in tqdm(models, desc='Model'):
        if not model['name'].startswith('15_'):
            continue

        target_model_name = {
            '15_vocab_01': '15a_vocab_01',
            '15_vocab_02': '15a_vocab_02',
            '15_predict_01': '15d_gpt_01',
        }[model['name']]

        assets = model_registry.get_assets(model['name'], model['version'])
        model_registry.register_model(target_model_name, model['version'])
    
        for asset in tqdm(assets, desc=f'Asset {model['name']}:{model['version']}', leave=False):
            ext = asset['maven2']['extension']
            
            if any(map(lambda x: ext.endswith(x), ('pom', '.md5', '.sha256', '.sha512', '.sha1'))):
                continue
                
            content = model_registry.get_asset_content(model['name'], model['version'], asset['maven2']['extension'], asset['maven2'].get('classifier', ''))
            
            with io.BytesIO(content) as b:
                model_registry.attach_asset(
                    model_name=target_model_name, 
                    model_version=model['version'], 
                    asset_ext=asset['maven2']['extension'], 
                    asset_classifier=asset['maven2'].get('classifier', ''),
                    asset=b)
finally:
    LOG.set_log_level('all', old_log_level)

Model:   0%|          | 0/144 [00:00<?, ?it/s]

Asset 15_vocab_01:124:   0%|          | 0/35 [00:00<?, ?it/s]

Asset 15_vocab_01:118:   0%|          | 0/35 [00:00<?, ?it/s]

Asset 15_vocab_01:119:   0%|          | 0/35 [00:00<?, ?it/s]

Asset 15_vocab_01:125:   0%|          | 0/35 [00:00<?, ?it/s]

...

Asset 15_vocab_02:14:   0%|          | 0/25 [00:00<?, ?it/s]

Asset 15_vocab_02:15:   0%|          | 0/25 [00:00<?, ?it/s]

Asset 15_vocab_02:13:   0%|          | 0/25 [00:00<?, ?it/s]

Asset 15_predict_01:1:   0%|          | 0/20 [00:00<?, ?it/s]

Asset 15_predict_01:2:   0%|          | 0/20 [00:00<?, ?it/s]